# DASHBOARD PYTHON AVANCEE


En utilisant le jeu de données `supermarket_sales.csv`, vous devez réaliser un tableau de bord interactif 
avec `Dash` qui permet d'analyser les ventes d'un supermarché.

Vous choisirez deux (2) des indicateurs et trois (3) des graphiques présentés ci-dessous dans 
votre tableau de bord. Chaque indicateur et graphique doit être interactif en fonction du **sexe** et de la **ville**. 

Indicateurs par sexe et par ville :

- Montant total des achats : Somme du montant total des achats (Total)
- Nombre total d'achats : Nombre total d'achats (Invoice ID)
- Évaluation moyenne : Moyenne des évaluations (Rating)

Graphiques :

- Histogramme donnant la répartition des montants totaux des achats par sexe et par ville.
- Diagramme en barres du nombre total d'achats par sexe et par ville.
- Diagramme circulaire montrant la répartition de la catégorie de produit (Product line) par sexe et par ville.
- Evolution du montant total des achats par semaine par ville.

## Importation des imports 

In [36]:
import pandas as pd 
import numpy as np 

# graphique 
import plotly.graph_objects as go
import plotly.express as px



## Importation, visualisation & compréhension des data 

In [19]:
data = pd.read_csv("data/supermarket_sales.csv")
display(data.head())

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,1/5/2019,13:08,Ewallet,522.83,4.761905,26.1415,9.1
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,3/8/2019,10:29,Cash,76.40,4.761905,3.8200,9.6
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.2155,340.5255,3/3/2019,13:23,Credit card,324.31,4.761905,16.2155,7.4
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.2880,489.0480,1/27/2019,20:33,Ewallet,465.76,4.761905,23.2880,8.4
4,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7,30.2085,634.3785,2/8/2019,10:37,Ewallet,604.17,4.761905,30.2085,5.3


In [20]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Invoice ID               1000 non-null   object 
 1   Branch                   1000 non-null   object 
 2   City                     1000 non-null   object 
 3   Customer type            1000 non-null   object 
 4   Gender                   1000 non-null   object 
 5   Product line             1000 non-null   object 
 6   Unit price               1000 non-null   float64
 7   Quantity                 1000 non-null   int64  
 8   Tax 5%                   1000 non-null   float64
 9   Total                    1000 non-null   float64
 10  Date                     1000 non-null   object 
 11  Time                     1000 non-null   object 
 12  Payment                  1000 non-null   object 
 13  cogs                     1000 non-null   float64
 14  gross margin percentage  

In [21]:
# Transformation du type en format data 
data["Date"] = pd.to_datetime(data["Date"], errors='coerce')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Invoice ID               1000 non-null   object        
 1   Branch                   1000 non-null   object        
 2   City                     1000 non-null   object        
 3   Customer type            1000 non-null   object        
 4   Gender                   1000 non-null   object        
 5   Product line             1000 non-null   object        
 6   Unit price               1000 non-null   float64       
 7   Quantity                 1000 non-null   int64         
 8   Tax 5%                   1000 non-null   float64       
 9   Total                    1000 non-null   float64       
 10  Date                     1000 non-null   datetime64[ns]
 11  Time                     1000 non-null   object        
 12  Payment                  1000 non-n

# Description des variables 

**Identifiants et Localisation**  
- Invoice ID : Le numéro de facture unique pour chaque transaction.
- Branch : La succursale du magasin (ici identifiée par des lettres : A, B, C).
- City : La ville où se situe le magasin.  
  
**Profil**    
-  ClientCustomer type : Le segment du client. "Member" pour ceux qui ont une carte de fidélité, "Normal" pour les autres.
- Gender : Le genre du client (Male/Female).Détails du ProduitProduct line : La catégorie du produit acheté (Santé et beauté, Accessoires électroniques, etc.).
- Unit price : Le prix d'une seule unité du produit.Quantity : Le nombre d'unités achetées.    


**Calculs Financiers**
- Tax 5% : Le montant de la taxe sur la valeur ajoutée (TVA) calculé sur l'achat ($5\%$).
- Total : Le prix final payé par le client (Prix unitaire $\times$ Quantité + Taxe).cogs : Signifie 
- Cost of Goods Sold (Coût des marchandises vendues). C'est ce que le produit a coûté au magasin pour l'acheter ou le fabriquer.
- gross margin percentage : Le pourcentage de marge brute. C'est le ratio de profitabilité.
- gross income : Le bénéfice brut généré par la vente ($Total - cogs - Taxe$ ou simplement la marge en valeur monétaire).  

**Logistique et Satisfaction**
- Date / Time : Le moment précis de la transaction.
- Payment : Le mode de paiement utilisé (Ewallet, Cash, Credit Card).
- Rating : La note de satisfaction donnée par le client sur une échelle de 1 à 10.

## Préparation des graphiques 

### Graphique: diagramme répartition des catégories de produits par sexe et par ville

In [37]:
# fonction qui calcul la répartition des catégories de produits par sexes
def repartition_cat_produits(data): 
    """
    La fonction permet de calculer le nombre de commande pour chaque catégorie de produit en fonction 
    du sexe et de la ville. 
    Elle prend en entrée un dataframe et renvoie en sorti un autre dataframe contenant le genre,
    le ville, la catégorie de produit et le nombre de commande. 
    """
    
    data_cat_prod = data.groupby(["Gender", "City", 'Product line']).size().reset_index()
    data_cat_prod = data_cat_prod.rename(columns = {0: "Nombre_commande"}) 
    
    return data_cat_prod 


In [38]:
repartition_cat_produits(data).head()

,Gender,City,Product line,Nombre_commande
0,Female,Mandalay,Electronic accessories,28
1,Female,Mandalay,Fashion accessories,33
2,Female,Mandalay,Food and beverages,29
3,Female,Mandalay,Health and beauty,20
4,Female,Mandalay,Home and lifestyle,22


In [50]:
# Graphique diagramme circulaire 
data_rep = repartition_cat_produits(data)
data_rep.columns

df_filtre = data_rep[(data_rep["City"] == "Mandalay") & (data_rep["Gender"] == "Female")]

graphe_cat_prod = px.pie(
    data_frame= df_filtre,
    values = "Nombre_commande",
    names = "Product line",
    title= "Répartition"
)

graphe_cat_prod.show()

## 